# Crop Disease Detection Model

This notebook documents the raw-image preprocessing, duplicate-safe split, 2,000-image-per-leaf-class balancing, GPU-only ResNet50 training, held-out evaluation, new-image inference, and Android TensorFlow Lite export. It produces the Keras model and reported metrics, then exports the TensorFlow Lite model used by the Flutter app.

Use the registered Python (CropCare GPU) kernel (Python 3.10, TensorFlow 2.10, TensorFlow DirectML) for training. The pipeline keeps RGB color, resizes to 224 x 224, uses ResNet50 channel normalization, and leaves background removal, grayscale, and Gaussian blur disabled to match the trained weights and preserve lesion color/detail.

## Phase 1 - Dataset preprocessing

This comparison intentionally starts with color-preserving inputs and no blur or segmentation.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageFilter

# Start Jupyter from the project folder, or adjust PROJECT_DIR if needed.
PROJECT_DIR = Path.cwd()
RAW_IMAGES_DIR = PROJECT_DIR / "raw images"
PREPROCESSED_DIR = PROJECT_DIR / "preprocessed"
TRAINING_IMAGES_DIR = PREPROCESSED_DIR / "rgb_resized_224"
IMAGE_SIZE = (224, 224)
MODEL_NAME = "ResNet50"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

# Optional transformations kept for controlled experiments; off by default.
USE_BACKGROUND_REMOVAL = False
USE_GRAYSCALE = False
USE_GAUSSIAN_FILTER = False
FORCE_REPROCESS = False

if not RAW_IMAGES_DIR.is_dir():
    raise FileNotFoundError(f"Dataset folder not found: {RAW_IMAGES_DIR}")

print(f"Project folder: {PROJECT_DIR.resolve()}")
print(f"Raw images folder: {RAW_IMAGES_DIR.resolve()}")
print(f"Training image folder: {TRAINING_IMAGES_DIR.resolve()}")
print(
    "Optional filters:",
    {"background_removal": USE_BACKGROUND_REMOVAL,
     "grayscale": USE_GRAYSCALE,
     "gaussian_filter": USE_GAUSSIAN_FILTER},
)
print(f"Force reprocess: {FORCE_REPROCESS}")


### 2. Inspect dataset and discover labels

In [ ]:
class_dirs = sorted(path for path in RAW_IMAGES_DIR.iterdir() if path.is_dir())
if not class_dirs:
    raise ValueError(f'No class folders found inside {RAW_IMAGES_DIR}')

records = []
for class_dir in class_dirs:
    files = sorted(path for path in class_dir.rglob('*')
                   if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)
    records.extend({'path': str(path), 'label': class_dir.name} for path in files)

image_df = pd.DataFrame(records, columns=['path', 'label'])
if image_df.empty:
    raise ValueError(f'No supported image files found inside {RAW_IMAGES_DIR}')
class_counts = image_df['label'].value_counts().sort_index()
print(f'Found {len(class_counts)} classes and {len(image_df):,} images')
display(class_counts.rename('image_count').to_frame())

# Save the discovered labels and source paths for traceability.
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)
image_df.to_csv(PREPROCESSED_DIR / 'source_image_manifest.csv', index=False)

### 3. Convert every source image to RGB


In [ ]:

def load_rgb_image(path):
    """Open an image, apply EXIF orientation, and convert it to three-channel RGB."""
    with Image.open(path) as image:
        return ImageOps.exif_transpose(image).convert("RGB")

rgb_sample = load_rgb_image(image_df.iloc[0]["path"])
print(f"RGB mode: {rgb_sample.mode}; channels: {len(rgb_sample.getbands())}")


### 4. Resize images to 224 x 224


In [ ]:
RESIZE_INTERPOLATION = Image.Resampling.BILINEAR

def resize_rgb_image(image, size=IMAGE_SIZE):
    """Resize RGB input to the fixed dimensions used by the Android app."""
    return image.resize(size, RESIZE_INTERPOLATION)

resize_sample = resize_rgb_image(rgb_sample)
print(f"Resized image dimensions: {resize_sample.size}")


### 5. Optional background removal with Otsu thresholding


In [ ]:

def remove_background_otsu(image):
    """Segment the leaf with Otsu thresholding and close small gaps in its mask."""
    pixels = np.array(image.convert("RGB"), dtype=np.uint8, copy=True)
    gray = np.asarray(image.convert("L"), dtype=np.uint8)
    histogram = np.bincount(gray.ravel(), minlength=256).astype(np.float64)
    probabilities = histogram / histogram.sum()
    levels = np.arange(256, dtype=np.float64)
    weight_background = np.cumsum(probabilities)
    mean_background = np.cumsum(probabilities * levels)
    mean_total = mean_background[-1]
    denominator = weight_background * (1.0 - weight_background)
    between_class_variance = np.zeros(256, dtype=np.float64)
    valid = denominator > 0
    between_class_variance[valid] = (
        (mean_total * weight_background[valid] - mean_background[valid]) ** 2
        / denominator[valid]
    )
    threshold = int(np.argmax(between_class_variance))
    mask = gray > threshold

    # The side selected by most border pixels is treated as background.
    border = np.concatenate((mask[0, :], mask[-1, :], mask[:, 0], mask[:, -1]))
    if border.mean() > 0.5:
        mask = ~mask

    mask_image = Image.fromarray((mask * 255).astype(np.uint8))
    mask_image = mask_image.filter(ImageFilter.MaxFilter(3)).filter(ImageFilter.MinFilter(3))
    mask = np.asarray(mask_image) > 0
    pixels[~mask] = 255
    return Image.fromarray(pixels, mode="RGB")

segmented_sample = remove_background_otsu(resize_sample)
print(f"Background-removed sample: {segmented_sample.mode}, {segmented_sample.size}")


### 6. Optional grayscale conversion


In [ ]:
def convert_to_grayscale(image):
    """Convert an RGB image to one luminance channel."""
    return image.convert("L")

grayscale_sample = convert_to_grayscale(segmented_sample)
print(f"Grayscale sample mode: {grayscale_sample.mode}; dimensions: {grayscale_sample.size}")


### 7. Optional Gaussian filtering


In [ ]:
def apply_gaussian_filter(image):
    """Apply the shared 3 x 3 Gaussian kernel [1,2,1;2,4,2;1,2,1] / 16."""
    gray = np.asarray(image.convert("L"), dtype=np.float32)
    padded = np.pad(gray, ((1, 1), (1, 1)), mode="edge")
    blurred = (
        padded[:-2, :-2] + 2 * padded[:-2, 1:-1] + padded[:-2, 2:]
        + 2 * padded[1:-1, :-2] + 4 * padded[1:-1, 1:-1] + 2 * padded[1:-1, 2:]
        + padded[2:, :-2] + 2 * padded[2:, 1:-1] + padded[2:, 2:]
    ) / 16.0
    return Image.fromarray(np.clip(np.rint(blurred), 0, 255).astype(np.uint8))

gaussian_sample = apply_gaussian_filter(grayscale_sample)
print(f"Gaussian-filtered sample mode: {gaussian_sample.mode}; kernel: 3 x 3")


### 8. Run the enabled preprocessing stages and save RGB inputs


In [ ]:
# Save only the final result of the enabled preprocessing chain.
TRAINING_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
processed_records = []
failed_images = []

for index, row in enumerate(image_df.itertuples(index=False), start=1):
    source_path = Path(row.path)
    class_dir = RAW_IMAGES_DIR / row.label
    relative_path = source_path.relative_to(class_dir).with_suffix(".png")
    output_path = TRAINING_IMAGES_DIR / row.label / relative_path
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.is_file() and not FORCE_REPROCESS:
        processed_records.append({
            "path": str(output_path),
            "label": row.label,
            "source_path": str(source_path),
        })
        if index % 500 == 0 or index == len(image_df):
            print(f"Reused {index:,} / {len(image_df):,} existing RGB images")
        continue

    try:
        image = resize_rgb_image(load_rgb_image(source_path))
        if USE_BACKGROUND_REMOVAL:
            image = remove_background_otsu(image)
        if USE_GRAYSCALE:
            image = convert_to_grayscale(image)
        if USE_GAUSSIAN_FILTER:
            image = apply_gaussian_filter(image)
        image = image.convert("RGB")
        image.save(output_path, format="PNG", optimize=True)
        processed_records.append({
            "path": str(output_path),
            "label": row.label,
            "source_path": str(source_path),
        })
    except Exception as exc:
        failed_images.append({"path": str(source_path), "error": str(exc)})

    if index % 500 == 0 or index == len(image_df):
        print(f"Processed {index:,} / {len(image_df):,} source images")

preprocessed_df = pd.DataFrame(processed_records)
preprocessed_df.to_csv(PREPROCESSED_DIR / "preprocessing_manifest.csv", index=False)
print(f"Successfully preprocessed: {len(preprocessed_df):,} / {len(image_df):,}")
print(f"Final RGB training images: {TRAINING_IMAGES_DIR.resolve()}")

if failed_images:
    failed_df = pd.DataFrame(failed_images)
    failed_df.to_csv(PREPROCESSED_DIR / "failed_images.csv", index=False)
    display(failed_df.head())
if preprocessed_df.empty:
    raise RuntimeError("No images were preprocessed successfully.")


### 9. Validate saved preprocessing outputs


In [ ]:
invalid_images = []
all_paths = preprocessed_df["path"].tolist()
missing_paths = [path for path in all_paths if not Path(path).is_file()]
if missing_paths:
    raise RuntimeError(f"{len(missing_paths)} processed images are missing.")

# The initial run checks every image; repeated runs recheck a spread of samples.
validation_paths = (
    all_paths if FORCE_REPROCESS
    else all_paths[::max(1, len(all_paths) // 200)]
)
for image_path in validation_paths:
    try:
        with Image.open(image_path) as image:
            if image.size != IMAGE_SIZE or image.mode != "RGB":
                invalid_images.append({
                    "path": image_path,
                    "size": image.size,
                    "mode": image.mode,
                    "expected_size": IMAGE_SIZE,
                    "expected_mode": "RGB",
                })
    except Exception as exc:
        invalid_images.append({"path": image_path, "error": str(exc)})

if invalid_images:
    invalid_df = pd.DataFrame(invalid_images)
    invalid_df.to_csv(PREPROCESSED_DIR / "invalid_preprocessed_images.csv", index=False)
    raise RuntimeError(
        f"{len(invalid_images)} final outputs failed validation; "
        "see invalid_preprocessed_images.csv"
    )

print(
    f"Validation passed: {len(all_paths):,} files exist; "
    f"{len(validation_paths):,} images checked for RGB mode and dimensions."
)


### 10. Normalize with the matching ResNet50 preprocessing


In [ ]:
def normalize_pixels(pixels, model_name=MODEL_NAME):
    """Apply input scaling that matches the selected classifier."""
    pixels = np.asarray(pixels, dtype=np.float32)
    if model_name == "CustomCNN":
        return pixels / 255.0
    if model_name == "ResNet50":
        try:
            from tensorflow.keras.applications.resnet50 import preprocess_input
            return preprocess_input(pixels)
        except ModuleNotFoundError:
            # ResNet50 caffe preprocessing: RGB to BGR, then subtract ImageNet means.
            return pixels[..., ::-1] - np.asarray(
                [103.939, 116.779, 123.68], dtype=np.float32
            )
    raise ValueError("Use CustomCNN or ResNet50.")

def load_normalized_image(path, model_name=MODEL_NAME):
    """Load an RGB image and normalize it for the model."""
    with Image.open(path) as image:
        pixels = np.asarray(image.convert("RGB"), dtype=np.float32)
    return normalize_pixels(pixels, model_name)

sample_image = load_normalized_image(preprocessed_df.iloc[0]["path"], MODEL_NAME)
print(f"Model preprocessing: {MODEL_NAME}")
print(f"RGB model input shape: {sample_image.shape}")
print(f"Normalized value range: {sample_image.min():.3f} to {sample_image.max():.3f}")
if not np.isfinite(sample_image).all():
    raise ValueError("Model-preprocessed pixels contain a non-finite value.")


## Phase 2 - Build a balanced training set and assess the GPU model

### 11. Create the duplicate-safe train, validation, and test splits

In [ ]:
import hashlib
import os

SEED = 42
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.15

if not np.isclose(TRAIN_FRACTION + VALIDATION_FRACTION + TEST_FRACTION, 1.0):
    raise ValueError("Train, validation, and test fractions must sum to 1.")

split_data = preprocessed_df.copy()
def normalized_path(path):
    return os.path.normcase(os.path.abspath(str(path)))

split_path = PREPROCESSED_DIR / "data_split.csv"
previous_assignments = {}
previous_duplicate_groups = {}
if split_path.is_file():
    previous = pd.read_csv(split_path)
    if {"source_path", "split"}.issubset(previous.columns):
        previous_assignments = {
            normalized_path(row.source_path): row.split
            for row in previous[["source_path", "split"]].itertuples(index=False)
        }
    if {"source_path", "duplicate_group"}.issubset(previous.columns):
        previous_duplicate_groups = {
            normalized_path(row.source_path): row.duplicate_group
            for row in previous[["source_path", "duplicate_group"]].itertuples(index=False)
        }

mapped_splits = split_data["source_path"].map(
    lambda path: previous_assignments.get(normalized_path(path))
)
mapped_duplicate_groups = split_data["source_path"].map(
    lambda path: previous_duplicate_groups.get(normalized_path(path))
)
if (
    len(previous_assignments)
    and mapped_splits.notna().all()
    and mapped_duplicate_groups.notna().all()
):
    split_data["split"] = mapped_splits
    split_data["duplicate_group"] = mapped_duplicate_groups
    print("Reusing the previous duplicate-safe split assignments for a fair comparison.")
else:
    print("No complete prior split found; creating a new deterministic duplicate-safe split.")
    split_data["duplicate_group"] = [
        hashlib.sha256(Path(path).read_bytes()).hexdigest()
        for path in split_data["source_path"]
    ]
    label_conflicts = split_data.groupby("duplicate_group")["label"].nunique()
    if (label_conflicts > 1).any():
        raise ValueError("An identical source image appears under multiple class labels; review duplicate labels.")
    split_parts = {"train": [], "validation": [], "test": []}
    rng = np.random.default_rng(SEED)
    for label, class_frame in split_data.groupby("label", sort=True):
        duplicate_groups = [
            group for _, group in class_frame.groupby("duplicate_group", sort=False)
        ]
        if len(duplicate_groups) < 3:
            raise ValueError(f"Class {label!r} has fewer than 3 unique images.")
        rng.shuffle(duplicate_groups)
        n_images = len(class_frame)
        test_target = max(1, int(round(n_images * TEST_FRACTION)))
        validation_target = max(1, int(round(n_images * VALIDATION_FRACTION)))
        test_count = validation_count = 0
        for group_index, duplicate_group in enumerate(duplicate_groups):
            groups_left = len(duplicate_groups) - group_index
            if test_count < test_target and groups_left > 2:
                split_parts["test"].append(duplicate_group)
                test_count += len(duplicate_group)
            elif validation_count < validation_target and groups_left > 1:
                split_parts["validation"].append(duplicate_group)
                validation_count += len(duplicate_group)
            else:
                split_parts["train"].append(duplicate_group)
    split_data = pd.concat([
        pd.concat(split_parts[name], ignore_index=True).assign(split=name)
        for name in ("train", "validation", "test")
    ], ignore_index=True)

duplicate_split_counts = split_data.groupby("duplicate_group")["split"].nunique()
if (duplicate_split_counts > 1).any():
    raise ValueError("An exact duplicate image would cross train/validation/test splits.")

train_df = split_data.loc[split_data["split"] == "train"].reset_index(drop=True)
validation_df = split_data.loc[split_data["split"] == "validation"].reset_index(drop=True)
test_df = split_data.loc[split_data["split"] == "test"].reset_index(drop=True)
class_names = sorted(preprocessed_df["label"].unique().tolist())
label_to_index = {name: index for index, name in enumerate(class_names)}
index_to_label = {index: name for name, index in label_to_index.items()}
split_data.to_csv(split_path, index=False)

print(f"Classes: {len(class_names)}")
print(f"Train: {len(train_df):,} | Validation: {len(validation_df):,} | Test: {len(test_df):,}")
print("Exact duplicate raw images were kept within one split.")
display(split_data.groupby(["split", "label"]).size().unstack(fill_value=0))


### 12. Save the class list for reproducible training

Run this after the split cell. It writes the 16 labels discovered from the raw-image folders in the same order used by the saved model.

In [ ]:
import json
from pathlib import Path

CLASS_NAMES_PATH = PROJECT_DIR / "model_output" / "class_names.json"
CLASS_NAMES_PATH.parent.mkdir(parents=True, exist_ok=True)
class_names = sorted(preprocessed_df["label"].unique().tolist())
CLASS_NAMES_PATH.write_text(json.dumps(class_names, indent=2), encoding="utf-8")
assert len(class_names) == 16, f"Expected 16 leaf classes; found {len(class_names)}"
print(f"Saved {len(class_names)} labels to {CLASS_NAMES_PATH}")


### 13. Augment raw training images to 2,000 per leaf class

Augmentation sources are restricted to rows marked `train` in `data_split.csv`. Validation and test folders stay untouched. If a valid 2,000-per-class output already exists, this cell verifies it and leaves it in place instead of overwriting it.

In [ ]:
"""Create a leakage-safe, materialized 2,000-image training set per leaf class.

Only raw training-split images are used as augmentation sources. Validation and
test images remain the untouched preprocessed originals.
"""

from __future__ import annotations

import csv
import json
import random
from collections import Counter
from pathlib import Path

from PIL import Image, ImageEnhance, ImageOps


ROOT = PROJECT_DIR
PREPROCESSED = ROOT / "preprocessed"
SPLIT_PATH = PREPROCESSED / "data_split.csv"
CLASS_NAMES_PATH = ROOT / "model_output" / "class_names.json"
OUTPUT = PREPROCESSED / "balanced_2000" / "train"
TARGET_PER_CLASS = 2000
IMAGE_SIZE = (224, 224)
SEED = 42
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def load_rgb(path: Path) -> Image.Image:
    with Image.open(path) as opened:
        image = ImageOps.exif_transpose(opened).convert("RGB")
        return image.resize(IMAGE_SIZE, Image.Resampling.BILINEAR)


def augment(image: Image.Image, rng: random.Random) -> tuple[Image.Image, str]:
    operations: list[str] = []
    if rng.random() < 0.5:
        image = ImageOps.mirror(image)
        operations.append("horizontal_flip")
    if rng.random() < 0.5:
        image = ImageOps.flip(image)
        operations.append("vertical_flip")
    angle = rng.choice((0, 90, 180, 270))
    if angle:
        image = image.rotate(angle, resample=Image.Resampling.BILINEAR, expand=False)
        operations.append(f"rotate_{angle}")
    brightness = rng.uniform(0.88, 1.12)
    contrast = rng.uniform(0.88, 1.12)
    image = ImageEnhance.Brightness(image).enhance(brightness)
    image = ImageEnhance.Contrast(image).enhance(contrast)
    operations.extend((f"brightness_{brightness:.3f}", f"contrast_{contrast:.3f}"))
    return image, ";".join(operations)


def main() -> None:
    if not SPLIT_PATH.is_file() or not CLASS_NAMES_PATH.is_file():
        raise FileNotFoundError("The preprocessed split manifest or class names are missing.")
    if OUTPUT.exists():
        raise FileExistsError(
            f"Refusing to overwrite the existing augmented dataset: {OUTPUT}. "
            "Review or move that folder before rebuilding."
        )

    classes = json.loads(CLASS_NAMES_PATH.read_text(encoding="utf-8"))
    with SPLIT_PATH.open("r", encoding="utf-8-sig", newline="") as stream:
        rows = list(csv.DictReader(stream))
    if not rows or set(row["label"] for row in rows) != set(classes):
        raise ValueError("Split manifest labels do not exactly match the 16 leaf classes.")

    # Exact duplicate groups must never cross the train/validation/test boundary.
    group_splits: dict[str, set[str]] = {}
    for row in rows:
        group_splits.setdefault(row["duplicate_group"], set()).add(row["split"])
    leakage_groups = [group for group, splits in group_splits.items() if len(splits) > 1]
    if leakage_groups:
        raise ValueError(f"Found {len(leakage_groups)} duplicate groups across data splits.")

    training_rows: dict[str, list[dict[str, str]]] = {name: [] for name in classes}
    for row in rows:
        if row["split"] == "train":
            source = Path(row["source_path"])
            if not source.is_file():
                raise FileNotFoundError(f"Raw training image is missing: {source}")
            training_rows[row["label"]].append(row)

    OUTPUT.mkdir(parents=True, exist_ok=False)
    manifest_path = OUTPUT.parent / "augmentation_manifest.csv"
    manifest_rows: list[dict[str, object]] = []
    summary: dict[str, object] = {
        "target_images_per_leaf_class": TARGET_PER_CLASS,
        "leaf_class_count": len(classes),
        "total_leaf_training_images": TARGET_PER_CLASS * len(classes),
        "image_size": list(IMAGE_SIZE),
        "seed": SEED,
        "source": "raw images referenced by the train rows in preprocessed/data_split.csv",
        "validation_and_test_augmented": False,
        "augmentation": [
            "horizontal/vertical flips",
            "random 0/90/180/270-degree rotations",
            "brightness factor 0.88 to 1.12",
            "contrast factor 0.88 to 1.12",
        ],
        "classes": {},
    }

    for class_index, class_name in enumerate(classes):
        source_rows = training_rows[class_name]
        if not source_rows:
            raise ValueError(f"No raw training images found for {class_name}.")
        if len(source_rows) > TARGET_PER_CLASS:
            raise ValueError(
                f"{class_name} has {len(source_rows)} training images, more than the target; "
                "the script will not discard source images silently."
            )

        class_dir = OUTPUT / class_name
        class_dir.mkdir(parents=True)
        rng = random.Random(SEED + class_index * 1009)
        generated = TARGET_PER_CLASS - len(source_rows)
        class_manifest: list[dict[str, object]] = []

        for index, row in enumerate(source_rows):
            image = load_rgb(Path(row["source_path"]))
            destination = class_dir / f"original_{index:05d}.jpg"
            image.save(destination, format="JPEG", quality=95, subsampling=0, optimize=True)
            class_manifest.append({
                "class": class_name,
                "split": "train",
                "kind": "resized_original",
                "source_path": row["source_path"],
                "output_path": str(destination.resolve()),
                "operations": "resize_224x224",
            })

        for index in range(generated):
            source_row = rng.choice(source_rows)
            image = load_rgb(Path(source_row["source_path"]))
            image, operations = augment(image, rng)
            destination = class_dir / f"augmented_{index:05d}.jpg"
            image.save(destination, format="JPEG", quality=95, subsampling=0, optimize=True)
            class_manifest.append({
                "class": class_name,
                "split": "train",
                "kind": "augmented",
                "source_path": source_row["source_path"],
                "output_path": str(destination.resolve()),
                "operations": operations,
            })

        saved_images = [p for p in class_dir.iterdir() if p.suffix.lower() in IMAGE_SUFFIXES]
        if len(saved_images) != TARGET_PER_CLASS:
            raise RuntimeError(f"{class_name}: expected {TARGET_PER_CLASS} files, found {len(saved_images)}.")
        manifest_rows.extend(class_manifest)
        summary["classes"][class_name] = {
            "raw_dataset_images": sum(
                1 for r in rows if r["label"] == class_name and Path(r["source_path"]).is_file()
            ),
            "original_training_images": len(source_rows),
            "generated_augmented_images": generated,
            "training_images_after_augmentation": len(saved_images),
        }
        print(
            f"{class_name}: {len(source_rows)} train originals + {generated} augmented = "
            f"{len(saved_images)}",
            flush=True,
        )

    with manifest_path.open("w", encoding="utf-8-sig", newline="") as stream:
        writer = csv.DictWriter(
            stream,
            fieldnames=("class", "split", "kind", "source_path", "output_path", "operations"),
        )
        writer.writeheader()
        writer.writerows(manifest_rows)
    (OUTPUT.parent / "augmentation_summary.json").write_text(
        json.dumps(summary, indent=2), encoding="utf-8"
    )
    print(
        f"Verified {sum(1 for _ in OUTPUT.rglob('*.jpg'))} images across "
        f"{len(classes)} leaf classes. Validation/test source images were not modified.",
        flush=True,
    )



SUMMARY_PATH = OUTPUT.parent / "augmentation_summary.json"
if OUTPUT.exists():
    if not SUMMARY_PATH.is_file():
        raise RuntimeError(f"Existing output has no audit summary: {SUMMARY_PATH}")
    saved_summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
    saved_counts = {
        folder.name: sum(path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES for path in folder.iterdir())
        for folder in OUTPUT.iterdir() if folder.is_dir()
    }
    if (
        len(saved_counts) != 16
        or any(count != TARGET_PER_CLASS for count in saved_counts.values())
        or saved_summary.get("total_leaf_training_images") != 32000
    ):
        raise RuntimeError("Existing balanced output failed its 2,000-per-class check; inspect it before rebuilding.")
    print("Verified existing augmentation output; skipped regeneration to avoid overwriting it.")
else:
    main()

augmentation_summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
print("Leaf classes:", augmentation_summary["leaf_class_count"])
print("Materialized training images:", augmentation_summary["total_leaf_training_images"])


### 14. Verify class counts and split isolation

In [ ]:
import json
from pathlib import Path
import pandas as pd

AUGMENTATION_SUMMARY_PATH = PROJECT_DIR / "preprocessed" / "balanced_2000" / "augmentation_summary.json"
augmentation_summary = json.loads(AUGMENTATION_SUMMARY_PATH.read_text(encoding="utf-8"))
class_balance = pd.DataFrame.from_dict(augmentation_summary["classes"], orient="index")
class_balance.index.name = "leaf_class"
assert len(class_balance) == 16
assert (class_balance["training_images_after_augmentation"] == 2000).all()
assert int(class_balance["training_images_after_augmentation"].sum()) == 32000
assert augmentation_summary["validation_and_test_augmented"] is False
display(class_balance)
print("Verified exactly 2,000 training images in each of 16 leaf classes; validation and test were not augmented.")


## Phase 3 - GPU-only ResNet50 training and held-out evaluation

Run these cells in order. The training code checks that TensorFlow places real matrix operations and trainable variables on the NVIDIA GPU; it stops instead of silently falling back to CPU. The negative-image gate is calibrated from validation data, while the final test set is used only once for reporting.

### 15. Load the GPU runtime and validate the data

This cell imports TensorFlow DirectML, checks a GPU matrix operation, confirms the 16 labels and checks source image paths.

In [ ]:
"""GPU-only retraining for the CropCare leaf disease classifier.

Run with the Python 3.10 environment containing TensorFlow 2.10 and the
TensorFlow DirectML plugin. This script stops if it cannot verify GPU compute.
"""

from __future__ import annotations

import csv
import hashlib
import json
import os
import random
import shutil
import time
from collections import Counter
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)


ROOT = PROJECT_DIR
PREPROCESSED = ROOT / "preprocessed"
OUTPUT = ROOT / "model_output" / "retrained_balanced_2000_gpu"
BALANCED_TRAIN = PREPROCESSED / "balanced_2000" / "train"
EXTERNAL = ROOT / "external_data" / "imagenette2-160"
CLASS_NAMES_PATH = ROOT / "model_output" / "class_names.json"
SIZE = 224
BACKBONE_SIZE = 160
SEED = 42
NOT_LEAF = "__not_a_supported_leaf__"
NUM_LEAF_CLASSES = 16
NUM_CLASSES = NUM_LEAF_CLASSES + 1
SAMPLES_PER_CLASS_PER_EPOCH = int(os.environ.get("CROP_SAMPLES_PER_CLASS", "2000"))
HEAD_EPOCHS = int(os.environ.get("CROP_HEAD_EPOCHS", "8"))
FINETUNE_EPOCHS = int(os.environ.get("CROP_FINETUNE_EPOCHS", "10"))
HEAD_BATCH = int(os.environ.get("CROP_HEAD_BATCH", "16"))
FINETUNE_BATCH = int(os.environ.get("CROP_FINETUNE_BATCH", "8"))
AUTOTUNE = tf.data.AUTOTUNE

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
OUTPUT.mkdir(parents=True, exist_ok=True)


def write_json(path: Path, value: object) -> None:
    path.write_text(json.dumps(value, indent=2, default=str), encoding="utf-8")


def verify_gpu() -> dict:
    gpus = tf.config.list_physical_devices("GPU")
    if not gpus:
        raise RuntimeError(
            "No TensorFlow GPU device is visible. Training is stopped; CPU fallback is disabled."
        )
    # This matrix operation must be placed by TensorFlow on the GPU device.
    with tf.device("/GPU:0"):
        x = tf.ones((512, 512), dtype=tf.float32)
        y = tf.linalg.matmul(x, x)
        result = float(tf.reduce_sum(y).numpy())
    device = y.device
    if "GPU:0" not in device.upper() or not np.isfinite(result):
        raise RuntimeError(f"GPU placement could not be verified: tensor device was {device!r}.")
    info = {
        "tensorflow": tf.__version__,
        "gpu_name": "NVIDIA GeForce GTX 1650 via DirectML adapter 0",
        "gpu_devices": [device.name for device in gpus],
        "gpu_probe_tensor_device": device,
        "gpu_probe_result": result,
        "training_device": "/GPU:0",
        "cpu_training_fallback": False,
        "python": os.sys.version,
    }
    write_json(OUTPUT / "gpu_runtime.json", info)
    print("Verified GPU matrix operation on", device, flush=True)
    return info


gpu_info = verify_gpu()


def file_digest(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def supported_images(folder: Path) -> list[Path]:
    suffixes = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    return sorted(p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in suffixes)


def map_imagenette_class(directory: str) -> str:
    mapping = {
        "n01440764": "tench fish",
        "n02102040": "springer spaniel",
        "n02979186": "cassette player",
        "n03000684": "chainsaw",
        "n03028079": "church",
        "n03394916": "French horn",
        "n03417042": "garbage truck",
        "n03425413": "gas pump",
        "n03445777": "golf ball",
        "n03888257": "parachute",
    }
    if directory not in mapping:
        raise ValueError(f"Unexpected non-leaf dataset category: {directory}")
    return mapping[directory]


if not (PREPROCESSED / "data_split.csv").is_file():
    raise FileNotFoundError("Run the dataset preprocessing/split cells first; data_split.csv is missing.")
if not BALANCED_TRAIN.is_dir():
    raise FileNotFoundError(f"The materialized 2,000-image training folders are missing: {BALANCED_TRAIN}")
if not (EXTERNAL / "train").is_dir() or not (EXTERNAL / "val").is_dir():
    raise FileNotFoundError("Imagenette 160 data is missing from external_data/imagenette2-160.")

leaf_df = pd.read_csv(PREPROCESSED / "data_split.csv")
class_names = json.loads(CLASS_NAMES_PATH.read_text(encoding="utf-8"))
if len(class_names) != NUM_LEAF_CLASSES:
    raise ValueError(f"Expected 16 disease/healthy labels; found {len(class_names)}.")
if set(leaf_df["label"].unique()) != set(class_names):
    raise ValueError("Preprocessed labels do not match model_output/class_names.json.")
if not set(leaf_df["split"].unique()).issubset({"train", "validation", "test"}):
    raise ValueError("Unexpected data split value.")

label_to_index = {name: i for i, name in enumerate(class_names)}
not_leaf_index = NUM_LEAF_CLASSES
all_class_names = class_names + [NOT_LEAF]

leaf_paths_by_split: dict[str, dict[str, list[str]]] = {}
for split in ("train", "validation", "test"):
    current = leaf_df[leaf_df["split"] == split]
    leaf_paths_by_split[split] = {
        name: current.loc[current["label"] == name, "path"].astype(str).tolist()
        for name in class_names
    }
    missing = [p for group in leaf_paths_by_split[split].values() for p in group if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError(f"{len(missing)} preprocessed image paths are missing; first: {missing[0]}")



### 16. Replace only training leaves with augmented images and prepare non-leaf data

The training split uses the 2,000-image folders; leaf validation/test remain original. Imagenette provides a separate unsupported-image class.

In [ ]:
# Replace only the leaf training paths with the leakage-safe, materialized
# augmentations. Validation and test continue to use untouched split originals.
leaf_paths_by_split["train"] = {}
for name in class_names:
    class_dir = BALANCED_TRAIN / name
    files = supported_images(class_dir)
    if len(files) != 2000:
        raise ValueError(f"Expected exactly 2,000 images for {name}; found {len(files)} in {class_dir}.")
    leaf_paths_by_split["train"][name] = [str(path.resolve()) for path in files]


# Keep the external source's official validation split as the final OOD test set.
# A deterministic held-out part of its training split calibrates the rejection gate.
negative_train_by_group: dict[str, list[Path]] = {}
negative_calibration_by_group: dict[str, list[Path]] = {}
negative_test_by_group: dict[str, list[Path]] = {}
negative_manifest_rows: list[dict] = []
split_rng = np.random.default_rng(SEED)
train_root, test_root = EXTERNAL / "train", EXTERNAL / "val"
groups = sorted(p.name for p in train_root.iterdir() if p.is_dir())
if len(groups) != 10:
    raise ValueError(f"Expected 10 Imagenette non-leaf categories, found {len(groups)}.")

train_hashes: set[str] = set()
official_test_hashes: set[str] = set()
for group in groups:
    train_files = supported_images(train_root / group)
    official_test_files = supported_images(test_root / group)
    if not train_files or not official_test_files:
        raise ValueError(f"Imagenette category {group} is empty in train or val.")
    permuted = np.array(train_files, dtype=object)
    split_rng.shuffle(permuted)
    calibration_count = max(1, int(round(len(permuted) * 0.15)))
    calibration_files = [Path(p) for p in permuted[:calibration_count]]
    model_train_files = [Path(p) for p in permuted[calibration_count:]]
    negative_train_by_group[group] = model_train_files
    negative_calibration_by_group[group] = calibration_files
    negative_test_by_group[group] = official_test_files
    for split, files, hashes in (
        ("train", model_train_files, train_hashes),
        ("validation", calibration_files, set()),
        ("test", official_test_files, official_test_hashes),
    ):
        for path in files:
            digest = file_digest(path)
            if split == "train":
                hashes.add(digest)
            if split == "test":
                hashes.add(digest)
            negative_manifest_rows.append({
                "path": str(path.resolve()),
                "label": NOT_LEAF,
                "source_category": map_imagenette_class(group),
                "split": split,
                "sha256": digest,
            })
if train_hashes.intersection(official_test_hashes):
    raise ValueError("Exact duplicate Imagenette files cross the training/test boundary.")
pd.DataFrame(negative_manifest_rows).to_csv(OUTPUT / "invalid_image_split.csv", index=False)


# Save the equal-per-class epoch design as a transparent class balance record.
class_train_counts = {name: len(leaf_paths_by_split["train"][name]) for name in class_names}
class_train_counts[NOT_LEAF] = sum(map(len, negative_train_by_group.values()))
balance_rows = [
    {
        "class": name,
        "original_training_images": count,
        "samples_per_balanced_epoch": SAMPLES_PER_CLASS_PER_EPOCH,
        "sampling_multiplier_per_epoch": round(SAMPLES_PER_CLASS_PER_EPOCH / count, 3),
        "leaf_augmentation_materialized": name in class_names,
        "negative_online_augmentation": name == NOT_LEAF,
    }
    for name, count in class_train_counts.items()
]
pd.DataFrame(balance_rows).to_csv(OUTPUT / "balanced_epoch_manifest.csv", index=False)




### 17. Build balanced TensorFlow input streams

The pipeline samples 2,000 images per output class in every epoch, applies online negative-image augmentation, and keeps validation/test deterministic.

In [ ]:
def make_path_stream(paths: list[str | Path], label: int, augment: bool):
    if not paths:
        raise ValueError(f"Cannot build an empty data stream for label {all_class_names[label]}.")
    path_ds = tf.data.Dataset.from_tensor_slices([str(p) for p in paths])
    path_ds = path_ds.shuffle(
        min(max(len(paths), 1), 2048), seed=SEED + label, reshuffle_each_iteration=True
    ).repeat()

    def load(path):
        encoded = tf.io.read_file(path)
        image = tf.image.decode_image(encoded, channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, (SIZE, SIZE), method=tf.image.ResizeMethod.BILINEAR)
        if augment:
            image = tf.image.random_flip_left_right(image)
            image = tf.image.random_flip_up_down(image)
            rotations = tf.random.uniform([], minval=0, maxval=4, dtype=tf.int32)
            image = tf.image.rot90(image, rotations)
            image = tf.image.random_brightness(image, max_delta=18.0)
            image = tf.image.random_contrast(image, lower=0.85, upper=1.15)
            image = tf.clip_by_value(image, 0.0, 255.0)
        # Keras ResNet50 Caffe preprocessing: RGB->BGR, then channel means.
        image = tf.reverse(image, axis=[-1])
        image = image - tf.constant([103.939, 116.779, 123.68], dtype=tf.float32)
        return image, tf.constant(label, dtype=tf.int32)

    return path_ds.map(load, num_parallel_calls=AUTOTUNE, deterministic=not augment)


def build_balanced_training_dataset(seed_offset: int = 0):
    """Build an epoch containing exactly the requested number of examples/class."""
    paths: list[str] = []
    labels: list[int] = []
    class_counts: Counter[int] = Counter()
    for label, name in enumerate(class_names):
        class_paths = leaf_paths_by_split["train"][name]
        if len(class_paths) != 2000:
            raise ValueError(f"{name}: expected 2,000 materialized training images, found {len(class_paths)}.")
        paths.extend(class_paths)
        labels.extend([label] * len(class_paths))
        class_counts[label] += len(class_paths)

    # Keep every negative source category available across training stages;
    # draw a stratified 2,000 distinct examples for this stage, then augment
    # them online so the finite pool is not reduced to a fixed subset forever.
    negative_groups = sorted(negative_train_by_group.items())
    if not negative_groups:
        raise ValueError("No non-leaf training groups were found.")
    quotient, remainder = divmod(SAMPLES_PER_CLASS_PER_EPOCH, len(negative_groups))
    rng = np.random.default_rng(SEED + seed_offset)
    for group_index, (_, group_paths) in enumerate(negative_groups):
        take = quotient + (1 if group_index < remainder else 0)
        if len(group_paths) < take:
            raise ValueError(f"Negative source category is too small for stratified sampling: {len(group_paths)} < {take}.")
        chosen = rng.choice(np.asarray(group_paths, dtype=object), size=take, replace=False)
        paths.extend(str(Path(path).resolve()) for path in chosen)
        labels.extend([not_leaf_index] * take)
        class_counts[not_leaf_index] += take

    expected = {index: SAMPLES_PER_CLASS_PER_EPOCH for index in range(NUM_CLASSES)}
    if dict(class_counts) != expected:
        raise RuntimeError(f"Training epoch is not exactly balanced: {dict(class_counts)}")
    number_of_examples = SAMPLES_PER_CLASS_PER_EPOCH * NUM_CLASSES
    ds = tf.data.Dataset.from_tensor_slices((paths, np.asarray(labels, dtype=np.int32)))
    ds = ds.shuffle(
        buffer_size=number_of_examples,
        seed=SEED + seed_offset,
        reshuffle_each_iteration=True,
    )

    def load_training(path, label):
        encoded = tf.io.read_file(path)
        image = tf.image.decode_image(encoded, channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, (SIZE, SIZE), method=tf.image.ResizeMethod.BILINEAR)

        def augment_negative():
            result = tf.image.random_flip_left_right(image)
            result = tf.image.random_flip_up_down(result)
            result = tf.image.random_brightness(result, max_delta=18.0)
            result = tf.image.random_contrast(result, lower=0.85, upper=1.15)
            return tf.clip_by_value(result, 0.0, 255.0)

        image = tf.cond(tf.equal(label, not_leaf_index), augment_negative, lambda: image)
        # Keras ResNet50 Caffe preprocessing: RGB->BGR, then channel means.
        image = tf.reverse(image, axis=[-1])
        image = image - tf.constant([103.939, 116.779, 123.68], dtype=tf.float32)
        return image, label

    ds = ds.map(load_training, num_parallel_calls=AUTOTUNE, deterministic=False)
    # Explicit repetition is required because fit uses a fixed steps_per_epoch.
    # It prevents Keras from exhausting the finite balanced examples after epoch 1.
    return ds.batch(HEAD_BATCH).repeat().prefetch(2), number_of_examples


def build_finetune_dataset():
    return build_balanced_training_dataset(seed_offset=1)


def finite_dataset(records: list[tuple[str, int]], batch_size: int):
    paths = [record[0] for record in records]
    labels = np.asarray([record[1] for record in records], dtype=np.int32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def load(path, label):
        encoded = tf.io.read_file(path)
        image = tf.image.decode_image(encoded, channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, (SIZE, SIZE), method=tf.image.ResizeMethod.BILINEAR)
        image = tf.reverse(image, axis=[-1])
        image = image - tf.constant([103.939, 116.779, 123.68], dtype=tf.float32)
        return image, label

    return ds.map(load, num_parallel_calls=AUTOTUNE).batch(batch_size).prefetch(1)


def leaf_records(split: str) -> list[tuple[str, int]]:
    return [
        (path, label_to_index[name])
        for name in class_names
        for path in leaf_paths_by_split[split][name]
    ]


def negative_records(index: dict[str, list[Path]]) -> list[tuple[str, int]]:
    return [(str(path), not_leaf_index) for files in index.values() for path in files]


validation_records = leaf_records("validation") + negative_records(negative_calibration_by_group)
test_records = leaf_records("test") + negative_records(negative_test_by_group)
validation_ds = finite_dataset(validation_records, 32)
test_ds = finite_dataset(test_records, 32)
balanced_train_ds, balanced_epoch_examples = build_balanced_training_dataset()


with tf.device("/GPU:0"):
    backbone = tf.keras.applications.ResNet50(
        include_top=False,
        weights="imagenet",
        input_shape=(BACKBONE_SIZE, BACKBONE_SIZE, 3),
    )
    backbone.trainable = False
    inputs = tf.keras.Input(shape=(SIZE, SIZE, 3), dtype=tf.float32, name="resnet_caffe_rgb")
    resized = tf.keras.layers.Resizing(
        BACKBONE_SIZE, BACKBONE_SIZE, interpolation="bilinear", name="backbone_resize"
    )(inputs)
    features = backbone(resized, training=False)
    pooled = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling")(features)
    x = tf.keras.layers.Dropout(0.30, name="head_dropout_1")(pooled)
    x = tf.keras.layers.Dense(256, activation="relu", name="classifier_dense")(x)
    x = tf.keras.layers.Dropout(0.25, name="head_dropout_2")(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", name="leaf_or_other")(x)
    model = tf.keras.Model(inputs, outputs, name="resnet50_balanced_leaf_gate")


trainable_devices = sorted({v.device for v in model.trainable_variables if v.device})
if not trainable_devices or not all("GPU:0" in device.upper() for device in trainable_devices):
    raise RuntimeError(f"Model trainable variables are not all on the verified GPU: {trainable_devices}")
write_json(OUTPUT / "training_config.json", {
    "seed": SEED,
    "input_size": [SIZE, SIZE],
    "resnet_internal_size": [BACKBONE_SIZE, BACKBONE_SIZE],
    "classes": all_class_names,
    "materialized_leaf_training_images_per_class": 2000,
    "materialized_leaf_training_total": 2000 * NUM_LEAF_CLASSES,
    "leaf_augmentation_source": "raw images referenced only by the training rows in data_split.csv",
    "validation_test_images_augmented": False,
    "not_leaf_original_training_pool": sum(map(len, negative_train_by_group.values())),
    "training_images_per_class_per_epoch": SAMPLES_PER_CLASS_PER_EPOCH,
    "balanced_examples_per_epoch": balanced_epoch_examples,
    "head_epochs_max": HEAD_EPOCHS,
    "fine_tune_epochs_max": FINETUNE_EPOCHS,
    "head_batch_size": HEAD_BATCH,
    "fine_tune_batch_size": FINETUNE_BATCH,
    "head_learning_rate": 0.001,
    "fine_tune_learning_rate": 0.00001,
    "unfrozen_resnet_block": "conv5_x only; all BatchNormalization layers frozen",
    "trainable_variable_devices": trainable_devices,
    "gpu": gpu_info,
})




### 18. Train the ResNet50 classifier head

The ImageNet ResNet50 backbone is frozen while the new 17-output head learns the crop and invalid-image labels.

In [ ]:
def callbacks(stage: str):
    best_weights = OUTPUT / f"{stage}_best.weights.h5"
    return [
        tf.keras.callbacks.ModelCheckpoint(
            str(best_weights), monitor="val_loss", save_best_only=True,
            save_weights_only=True, verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=3, restore_best_weights=True, verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1
        ),
        tf.keras.callbacks.BackupAndRestore(backup_dir=str(OUTPUT / f"{stage}_recovery")),
        tf.keras.callbacks.CSVLogger(str(OUTPUT / f"{stage}_keras_history.csv")),
    ]


def train_stage(stage: str, learning_rate: float, epochs: int, batch_size: int):
    global balanced_train_ds, balanced_epoch_examples
    if stage == "head":
        balanced_train_ds, balanced_epoch_examples = build_balanced_training_dataset()
    else:
        balanced_train_ds, balanced_epoch_examples = build_finetune_dataset()
    if batch_size != (HEAD_BATCH if stage == "head" else FINETUNE_BATCH):
        balanced_train_ds = balanced_train_ds.unbatch().batch(batch_size).prefetch(1)
    steps = balanced_epoch_examples // batch_size
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    print(
        f"Starting {stage}: GPU=/GPU:0, examples/epoch={balanced_epoch_examples}, "
        f"steps/epoch={steps}, batch={batch_size}, max_epochs={epochs}",
        flush=True,
    )
    started = time.time()
    with tf.device("/GPU:0"):
        history = model.fit(
            balanced_train_ds,
            validation_data=validation_ds,
            epochs=epochs,
            steps_per_epoch=steps,
            callbacks=callbacks(stage),
            verbose=2,
        )
    seconds = time.time() - started
    model.load_weights(OUTPUT / f"{stage}_best.weights.h5")
    model.save_weights(OUTPUT / f"{stage}_selected.weights.h5")
    print(f"Finished {stage} in {seconds / 60:.1f} minutes.", flush=True)
    return history, seconds


reuse_head_dir = Path(os.environ["CROP_REUSE_HEAD_DIR"]) if os.environ.get("CROP_REUSE_HEAD_DIR") else None
head_weights = OUTPUT / "head_selected.weights.h5"
if reuse_head_dir is not None:
    source_weights = reuse_head_dir / "head_selected.weights.h5"
    source_history = reuse_head_dir / "head_keras_history.csv"
    if not source_weights.is_file() or not source_history.is_file():
        raise FileNotFoundError(f"Reusable completed head-stage artifacts are incomplete: {reuse_head_dir}")
    model.load_weights(source_weights)
    shutil.copy2(source_weights, head_weights)
    shutil.copy2(source_history, OUTPUT / "head_keras_history.csv")
    head_frame = pd.read_csv(source_history)
    history_head = tf.keras.callbacks.History()
    history_head.history = {
        key: head_frame[key].astype(float).tolist()
        for key in ("loss", "accuracy", "val_loss", "val_accuracy")
        if key in head_frame
    }
    history_head.epoch = list(range(len(head_frame)))
    head_seconds = 0.0
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    print(f"Reusing verified {len(head_frame)}-epoch frozen-head training from {reuse_head_dir}", flush=True)
else:
    history_head, head_seconds = train_stage("head", 1e-3, HEAD_EPOCHS, HEAD_BATCH)
    head_weights = OUTPUT / "head_selected.weights.h5"
head_validation = model.evaluate(validation_ds, verbose=0, return_dict=True)



### 19. Unfreeze and fine-tune ResNet50 conv5_x

Only the final residual stage is unfrozen; BatchNormalization layers remain frozen. The selected weights are restored by validation loss.

In [ ]:
# Fine-tune the final residual stage. The stem and conv2_x through conv4_x stay frozen.
backbone.trainable = True
for layer in backbone.layers:
    layer.trainable = layer.name.startswith("conv5_")
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
fine_trainable_names = [layer.name for layer in backbone.layers if layer.trainable]
if not fine_trainable_names or not all(name.startswith("conv5_") for name in fine_trainable_names):
    raise RuntimeError("The intended ResNet50 conv5_x fine-tuning boundary was not applied.")
fine_trainable_devices = sorted({v.device for v in model.trainable_variables if v.device})
if not fine_trainable_devices or not all("GPU:0" in device.upper() for device in fine_trainable_devices):
    raise RuntimeError(f"Fine-tuning variables are not all on the verified GPU: {fine_trainable_devices}")
write_json(OUTPUT / "resnet_freeze_plan.json", {
    "frozen": "ResNet50 stem, conv2_x, conv3_x and conv4_x",
    "unfrozen": "ResNet50 conv5_x and disease head",
    "batch_normalization": "Frozen during both stages",
    "unfrozen_backbone_layer_names": fine_trainable_names,
    "fine_tuning_variable_devices": fine_trainable_devices,
})
history_fine, fine_seconds = train_stage("finetune", 1e-5, FINETUNE_EPOCHS, FINETUNE_BATCH)
fine_weights = OUTPUT / "finetune_selected.weights.h5"
fine_validation = model.evaluate(validation_ds, verbose=0, return_dict=True)

# Select the stage using validation loss only. The combined test set has not been used.
if fine_validation["loss"] < head_validation["loss"]:
    model.load_weights(fine_weights)
    selected_stage = "ResNet50 conv5_x fine-tuned"
    selected_validation = fine_validation
else:
    model.load_weights(head_weights)
    selected_stage = "Frozen ResNet50 feature extractor"
    selected_validation = head_validation
model.save(OUTPUT / "resnet50_leafgate.keras")
print("Selected stage:", selected_stage, "validation:", selected_validation, flush=True)




### 20. Calibrate the invalid-image gate and evaluate the untouched test set

This section records per-class scores, the two confusion matrices, and the exact predictions used by the report and app.

In [ ]:
def predict_records(dataset):
    probabilities = model.predict(dataset, verbose=0)
    return np.asarray(probabilities, dtype=np.float32)


validation_probabilities = predict_records(validation_ds)
validation_y = np.asarray([label for _, label in validation_records], dtype=np.int32)
test_probabilities = predict_records(test_ds)
test_y = np.asarray([label for _, label in test_records], dtype=np.int32)


def choose_gate_thresholds(y_true: np.ndarray, probabilities: np.ndarray):
    leaf_true = y_true != not_leaf_index
    other_probability = probabilities[:, not_leaf_index]
    leaf_confidence = probabilities[:, :NUM_LEAF_CLASSES].max(axis=1)
    top_class_is_leaf = np.argmax(probabilities, axis=1) != not_leaf_index
    candidates = np.linspace(0.0, 1.0, 101)
    rows = []
    best = None
    for other_cutoff in candidates:
        for leaf_cutoff in candidates:
            accept = (
                top_class_is_leaf
                & (other_probability < other_cutoff)
                & (leaf_confidence >= leaf_cutoff)
            )
            leaf_sensitivity = float(np.mean(accept[leaf_true])) if leaf_true.any() else 0.0
            other_specificity = float(np.mean(~accept[~leaf_true])) if (~leaf_true).any() else 0.0
            balanced_accuracy = (leaf_sensitivity + other_specificity) / 2.0
            rows.append({
                "not_leaf_probability_threshold": float(other_cutoff),
                "minimum_leaf_confidence": float(leaf_cutoff),
                "leaf_acceptance_sensitivity": leaf_sensitivity,
                "non_leaf_rejection_specificity": other_specificity,
                "balanced_accuracy": balanced_accuracy,
            })
            # Preserve at least 98% of known crop leaves, then maximize rejection.
            key = (leaf_sensitivity >= 0.98, other_specificity, leaf_sensitivity, balanced_accuracy)
            if best is None or key > best[0]:
                best = (key, float(other_cutoff), float(leaf_cutoff), leaf_sensitivity, other_specificity)
    pd.DataFrame(rows).to_csv(OUTPUT / "gate_threshold_sweep.csv", index=False)
    if best is None:
        raise RuntimeError("Could not calibrate the invalid-image decision threshold.")
    return {
        "not_leaf_probability_threshold": best[1],
        "minimum_leaf_confidence": best[2],
        "validation_leaf_acceptance_sensitivity": best[3],
        "validation_non_leaf_rejection_specificity": best[4],
        "selection_rule": "maximize non-leaf rejection while retaining >=98% validation leaf acceptance",
    }


gate = choose_gate_thresholds(validation_y, validation_probabilities)
write_json(OUTPUT / "invalid_image_gate.json", gate)


def gate_predictions(probabilities: np.ndarray) -> np.ndarray:
    return (
        (np.argmax(probabilities, axis=1) != not_leaf_index)
        &
        (probabilities[:, not_leaf_index] < gate["not_leaf_probability_threshold"])
        & (probabilities[:, :NUM_LEAF_CLASSES].max(axis=1) >= gate["minimum_leaf_confidence"])
    )


top_17 = np.argmax(test_probabilities, axis=1)
gate_accept = gate_predictions(test_probabilities)
gate_binary_truth = (test_y != not_leaf_index).astype(np.int32)  # positive = supported leaf
gate_binary_prediction = gate_accept.astype(np.int32)
gate_cm = confusion_matrix(gate_binary_truth, gate_binary_prediction, labels=[0, 1])
test_leaf_mask = test_y != not_leaf_index
test_negative_mask = ~test_leaf_mask
disease_only_prediction = np.argmax(test_probabilities[:, :NUM_LEAF_CLASSES], axis=1)
end_to_end_correct = np.where(
    test_leaf_mask,
    gate_accept & (disease_only_prediction == test_y),
    ~gate_accept,
)

gate_metrics = {
    "positive_definition": "supported crop leaf accepted by the gate",
    "negative_definition": "non-leaf input correctly rejected by the gate",
    "TP_leaf_accepted": int(gate_cm[1, 1]),
    "FN_leaf_rejected": int(gate_cm[1, 0]),
    "FP_nonleaf_accepted": int(gate_cm[0, 1]),
    "TN_nonleaf_rejected": int(gate_cm[0, 0]),
    "leaf_acceptance_recall": float(recall_score(gate_binary_truth, gate_binary_prediction, zero_division=0)),
    "nonleaf_rejection_specificity": float(gate_cm[0, 0] / max(gate_cm[0].sum(), 1)),
    "gate_precision_for_accepted_leaf": float(precision_score(gate_binary_truth, gate_binary_prediction, zero_division=0)),
    "gate_f1": float(f1_score(gate_binary_truth, gate_binary_prediction, zero_division=0)),
    "gate_balanced_accuracy": float(balanced_accuracy_score(gate_binary_truth, gate_binary_prediction)),
    "gate_confusion_matrix_labels": ["not supported leaf", "supported leaf"],
    "gate_confusion_matrix": gate_cm.tolist(),
}

full_cm = confusion_matrix(test_y, top_17, labels=np.arange(NUM_CLASSES))
test_report = classification_report(
    test_y,
    top_17,
    labels=np.arange(NUM_CLASSES),
    target_names=all_class_names,
    output_dict=True,
    zero_division=0,
)
accepted_leaf_total = int(test_leaf_mask.sum())
negative_total = int(test_negative_mask.sum())
test_metrics = {
    "selected_stage": selected_stage,
    "validation_loss": float(selected_validation["loss"]),
    "validation_accuracy": float(selected_validation["accuracy"]),
    "combined_test_examples": int(len(test_y)),
    "combined_17_way_accuracy": float(accuracy_score(test_y, top_17)),
    "end_to_end_accuracy_after_gate": float(np.mean(end_to_end_correct)),
    "leaf_test_examples": accepted_leaf_total,
    "nonleaf_test_examples": negative_total,
    "forced_16_disease_class_accuracy_on_leaf_test": float(accuracy_score(test_y[test_leaf_mask], disease_only_prediction[test_leaf_mask])),
    "system_leaf_correct_and_accepted_rate": float(np.mean((disease_only_prediction[test_leaf_mask] == test_y[test_leaf_mask]) & gate_accept[test_leaf_mask])),
    "test_macro_precision_17_way": float(test_report["macro avg"]["precision"]),
    "test_macro_recall_17_way": float(test_report["macro avg"]["recall"]),
    "test_macro_f1_17_way": float(test_report["macro avg"]["f1-score"]),
    "invalid_image_gate": gate_metrics,
}
write_json(OUTPUT / "test_metrics.json", test_metrics)
pd.DataFrame(full_cm, index=all_class_names, columns=all_class_names).to_csv(OUTPUT / "confusion_matrix_17_class.csv")
pd.DataFrame(test_report).T.to_csv(OUTPUT / "classification_report_17_class.csv")
pd.DataFrame(gate_cm, index=["actual_not_leaf", "actual_supported_leaf"], columns=["rejected_invalid", "accepted_leaf"]).to_csv(
    OUTPUT / "invalid_gate_confusion_matrix.csv"
)

prediction_rows = []
for (path, true_index), probabilities, predicted_index, accepted in zip(
    test_records, test_probabilities, top_17, gate_accept
):
    prediction_rows.append({
        "path": path,
        "actual_class": all_class_names[true_index],
        "predicted_17_way": all_class_names[int(predicted_index)],
        "gate_result": "leaf accepted" if accepted else "invalid/rejected",
        "not_leaf_probability": float(probabilities[not_leaf_index]),
        "top_leaf_probability": float(probabilities[:NUM_LEAF_CLASSES].max()),
        "top_leaf_class": class_names[int(np.argmax(probabilities[:NUM_LEAF_CLASSES]))],
        "correct_17_way": bool(predicted_index == true_index),
    })
pd.DataFrame(prediction_rows).to_csv(OUTPUT / "test_predictions.csv", index=False)




### 21. Save learning curves and export TensorFlow Lite

This cell writes the Keras model, float16-weight TFLite model, label order, app gate thresholds, and metadata.

In [ ]:
# Merge the two Keras histories into a single plot-ready epoch table.
metric_rows = []
global_epoch = 0
for stage_name, history in (("Frozen feature extractor", history_head), ("Fine-tuned conv5_x", history_fine)):
    epochs_in_history = len(history.history.get("loss", []))
    for i in range(epochs_in_history):
        global_epoch += 1
        metric_rows.append({
            "global_epoch": global_epoch,
            "stage": stage_name,
            "stage_epoch": i + 1,
            "loss": float(history.history["loss"][i]),
            "accuracy": float(history.history["accuracy"][i]),
            "val_loss": float(history.history["val_loss"][i]),
            "val_accuracy": float(history.history["val_accuracy"][i]),
        })
pd.DataFrame(metric_rows).to_csv(OUTPUT / "training_metrics.csv", index=False)


# Convert the validation-selected 17-class classifier for the Flutter app.
tflite_path = OUTPUT / "crop_disease_leaf_gate.tflite"
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_path.write_bytes(converter.convert())

interpreter = tf.lite.Interpreter(model_path=str(tflite_path), num_threads=2)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]
if list(input_details["shape"]) != [1, SIZE, SIZE, 3] or input_details["dtype"] != np.float32:
    raise RuntimeError(f"Unexpected TFLite input tensor: {input_details}")
if list(output_details["shape"]) != [1, NUM_CLASSES]:
    raise RuntimeError(f"Unexpected TFLite output tensor: {output_details}")
sample_x, _ = next(iter(test_ds.take(1)))
sample = sample_x[:1].numpy()
interpreter.set_tensor(input_details["index"], sample)
interpreter.invoke()
tflite_scores = interpreter.get_tensor(output_details["index"])[0]
keras_scores = model.predict(sample, verbose=0)[0]
if not np.isfinite(tflite_scores).all() or np.argmax(tflite_scores) != np.argmax(keras_scores):
    raise RuntimeError("TFLite export failed the finite-score/top-class parity check.")
write_json(OUTPUT / "tflite_validation.json", {
    "input_shape": input_details["shape"].tolist(),
    "output_shape": output_details["shape"].tolist(),
    "keras_top_label": all_class_names[int(np.argmax(keras_scores))],
    "tflite_top_label": all_class_names[int(np.argmax(tflite_scores))],
    "max_abs_score_difference": float(np.max(np.abs(keras_scores - tflite_scores))),
    "file_bytes": tflite_path.stat().st_size,
})

metadata = {
    "model_name": "CropCare ResNet50 supported-leaf and disease classifier",
    "format": "TensorFlow Lite",
    "input_shape": [1, SIZE, SIZE, 3],
    "input_dtype": "float32",
    "output_shape": [1, NUM_CLASSES],
    "class_names": all_class_names,
    "not_leaf_label": NOT_LEAF,
    "invalid_image_gate": gate,
    "selected_stage": selected_stage,
    "preprocessing": {
        "orientation": "EXIF correction in Flutter image decoder",
        "color": "RGB",
        "resize": [SIZE, SIZE],
        "interpolation": "bilinear",
        "normalization": "RGB to BGR then subtract [103.939, 116.779, 123.68]; no division by 255",
    },
    "resnet_internal_resize": [BACKBONE_SIZE, BACKBONE_SIZE],
    "training": {
        "balanced_sampling": "2,000 samples for each of 16 supported crop classes and the not-a-supported-leaf class per epoch",
        "materialized_leaf_dataset": "2,000 images per leaf class, generated from raw training-split images only",
        "not_leaf_pool": "Full training pool retained; 2,000 randomly sampled and augmented negatives per epoch",
        "samples_per_class_per_epoch": SAMPLES_PER_CLASS_PER_EPOCH,
        "augmentation": ["offline horizontal/vertical flips", "offline random 0/90/180/270-degree rotations", "offline brightness and contrast factors 0.88 to 1.12", "online augmentation for not-a-supported-leaf examples"],
        "seed": SEED,
        "test_not_used_for_threshold_selection": True,
        "gpu_device": "/GPU:0 DirectML",
    },
    "test_metrics": test_metrics,
    "warning": "Out-of-distribution rejection is learned from a finite negative-image set and cannot guarantee rejection of every possible image.",
}
write_json(OUTPUT / "model_metadata.json", metadata)
write_json(OUTPUT / "class_names.json", all_class_names)

print("Training and held-out evaluation finished.", flush=True)
print(json.dumps(test_metrics, indent=2), flush=True)
print("Artifacts:", OUTPUT, flush=True)


### 22. Copy the final model and metadata into the Flutter app

In [ ]:
import shutil

OUTPUT = PROJECT_DIR / "model_output" / "retrained_balanced_2000_gpu"
APP_ASSETS = PROJECT_DIR / "crop_disease_app" / "assets"
APP_ASSETS.mkdir(parents=True, exist_ok=True)
for source, destination in (
    (OUTPUT / "crop_disease_leaf_gate.tflite", APP_ASSETS / "crop_disease_resnet50.tflite"),
    (OUTPUT / "class_names.json", APP_ASSETS / "class_names.json"),
    (OUTPUT / "model_metadata.json", APP_ASSETS / "model_metadata.json"),
):
    if not source.is_file():
        raise FileNotFoundError(source)
    shutil.copy2(source, destination)
    print(f"Copied {source.name} to {destination}")


### 23. Review the saved held-out results

In [ ]:
import json
from pathlib import Path

OUTPUT = PROJECT_DIR / "model_output" / "retrained_balanced_2000_gpu"
metrics = json.loads((OUTPUT / "test_metrics.json").read_text(encoding="utf-8"))
print(f"Leaf-only accuracy: {metrics['forced_16_disease_class_accuracy_on_leaf_test']:.2%}")
print(f"Gated end-to-end Keras accuracy: {metrics['end_to_end_accuracy_after_gate']:.2%}")
print("Invalid-image gate (TN, FP; FN, TP):")
print(metrics["invalid_image_gate"]["gate_confusion_matrix"])
print("Run artifacts:", OUTPUT)


### 24. Predict a new image with the final 17-output model

Set `NEW_IMAGE_PATH` to a leaf image or an unrelated image. The same acceptance logic as the Android app rejects unsupported inputs and withholds disease advice.

In [ ]:
import io
import json
from pathlib import Path
import numpy as np
from PIL import Image, ImageOps
from IPython.display import display

OUTPUT = PROJECT_DIR / "model_output" / "retrained_balanced_2000_gpu"
MODEL_PATH = OUTPUT / "resnet50_leafgate.keras"
PREDICTION_CLASSES = json.loads((OUTPUT / "class_names.json").read_text(encoding="utf-8"))
MODEL_METADATA = json.loads((OUTPUT / "model_metadata.json").read_text(encoding="utf-8"))
GATE = MODEL_METADATA["invalid_image_gate"]
NOT_LEAF = "__not_a_supported_leaf__"
NEW_IMAGE_PATH = None  # Replace with a full path, for example Path("my_leaf.jpg")

if NEW_IMAGE_PATH is not None:
    prediction_model = tf.keras.models.load_model(MODEL_PATH, compile=False)
    image_path = Path(NEW_IMAGE_PATH)
    with Image.open(image_path) as source:
        original = ImageOps.exif_transpose(source).convert("RGB")
        resized = original.resize((224, 224), Image.Resampling.BILINEAR)
    pixels = np.asarray(resized, dtype=np.float32)
    model_input = tf.keras.applications.resnet50.preprocess_input(pixels)[None, ...]
    scores = prediction_model.predict(model_input, verbose=0)[0]
    if not np.isfinite(scores).all():
        raise ValueError("The model returned a non-finite score.")
    not_leaf_index = PREDICTION_CLASSES.index(NOT_LEAF)
    leaf_indices = [i for i, name in enumerate(PREDICTION_CLASSES) if name != NOT_LEAF]
    best_leaf_index = max(leaf_indices, key=lambda i: float(scores[i]))
    best_raw_index = int(np.argmax(scores))
    accepted = (
        best_raw_index != not_leaf_index
        and float(scores[not_leaf_index]) < float(GATE["not_leaf_probability_threshold"])
        and float(scores[best_leaf_index]) >= float(GATE["minimum_leaf_confidence"])
    )
    display(original)
    print("Image:", image_path.name)
    if not accepted:
        print("Invalid image: no supported crop leaf detected. No disease advice is shown.")
        print(f"Not-leaf score: {scores[not_leaf_index]:.4%}")
    else:
        print("Top disease/healthy predictions:")
        top_leaf = sorted(leaf_indices, key=lambda i: float(scores[i]), reverse=True)[:3]
        for rank, index in enumerate(top_leaf, start=1):
            label = PREDICTION_CLASSES[index].replace("___", " - ").replace("_", " ")
            print(f"{rank}. {label}: {scores[index]:.2%}")
else:
    print(f"Set NEW_IMAGE_PATH before inference. Model: {MODEL_PATH}")
